# 1. Get and tame NEM data

The **National Electricity Market (NEM)** is Australia's wholesale
electricity market, covering five interconnected regions across the
eastern seaboard and South Australia. Prices are set every five minutes
by a central dispatch engine — the world's fastest-clearing wholesale
energy market.

This notebook loads that data, builds intuition for what it looks like,
and produces the clean parquet files that every later notebook depends on.

## Objectives

- See the five NEM regions on a map and understand how they connect.
- Load dispatch-interval prices and demand via `grian.data`, understanding what it does under the hood.
- Inspect the raw data: columns, dtypes, timestamps, intervention flags.
- Visualise price and demand for SA1: full history, weekly cycles, daily shape.
- Compare all five regions side by side.
- Build and cache the final 5-min and 30-min parquet datasets.

## Prerequisites

- First notebook — nothing prior required.
- Internet access for the initial AEMO download (cached locally after that).

In [ ]:
import logging
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from grian.config import load_config, repo_root
from grian.viz import apply_style, save_fig

cfg = load_config()
apply_style()

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
warnings.filterwarnings("ignore", category=FutureWarning)

---
## 1. The NEM regions

The NEM covers five regions, each corresponding roughly to a state.
They are connected by high-voltage **interconnectors** — when one region
has cheap power and another is short, electricity flows between them.
But the interconnectors have limited capacity, so prices can diverge
dramatically between regions.

| Region ID | State | Character |
|---|---|---|
| `NSW1` | New South Wales (incl. ACT) | Largest demand, coal + solar |
| `QLD1` | Queensland | Coal + large-scale solar, gas peakers |
| `VIC1` | Victoria | Brown coal baseload, growing wind |
| `SA1` | South Australia | Wind + solar dominant, volatile prices |
| `TAS1` | Tasmania | Hydro-dominated, Basslink interconnector |

In [ ]:
# Load the NEM region boundaries (bundled GeoJSON)
regions_gdf = gpd.read_file(repo_root() / "data" / "nem_regions.geojson")
regions_gdf

In [ ]:
# Map the five NEM regions
REGION_COLORS = {
    "NSW1": "#2196F3",
    "QLD1": "#FF9800",
    "VIC1": "#4CAF50",
    "SA1": "#F44336",
    "TAS1": "#9C27B0",
}

fig, ax = plt.subplots(figsize=(8, 10))
for _, row in regions_gdf.iterrows():
    regions_gdf[regions_gdf["nem_region"] == row["nem_region"]].plot(
        ax=ax, color=REGION_COLORS[row["nem_region"]],
        edgecolor="white", linewidth=1.5, alpha=0.7,
    )
    ax.annotate(
        f"{row['nem_region']}\n{row['nem_label']}",
        xy=(row["label_lon"], row["label_lat"]),
        ha="center", fontsize=9, fontweight="bold",
    )

ax.set_title("National Electricity Market — five regions", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(112, 155)
ax.set_ylim(-45, -10)
save_fig(fig, "01_nem_regions_map")
plt.show()

---
## 2. Loading data with `grian.data`

Under the hood, `grian.data.load_prices` calls
[NEMOSIS](https://github.com/UNSW-CEEM/NEMOSIS) `dynamic_data_compiler`
to pull AEMO's public archive. Two things happen automatically that you
should know about:

### Timestamp shift (interval-ending → interval-start)

AEMO timestamps are **interval-ending**: a row stamped
`2024-01-01 00:05:00` covers the five minutes *before* that time
(`00:00:00` to `00:04:59`). Weather data, by convention, uses
interval-beginning. If you join price and weather without correcting
this, every feature is shifted by one interval — a subtle leak.

`load_prices` subtracts one interval (5 min for dispatch, 30 min for
trading) so that every timestamp in this project means **start of
interval**. This happens once here and is never repeated.

### Intervention filtering

During market interventions (e.g. AEMO directs a generator to run),
AEMO publishes **two rows** for the same timestamp: one with
`INTERVENTION=0` (the counterfactual dispatch) and one with
`INTERVENTION=1` (the actual intervention outcome). We keep only
`INTERVENTION=0` — the normal market clearing — to avoid duplicates.

### What columns come back

| Function | Source table | Raw column | Returned as | Unit |
|---|---|---|---|---|
| `load_prices` | `DISPATCHPRICE` | `RRP` (Regional Reference Price) | `price` | $/MWh |
| `load_demand` | `DISPATCHREGIONSUM` | `TOTALDEMAND` | `demand` | MW |

In [ ]:
from grian.data import build_dataset, load_demand, load_prices

REGION = cfg["region"]  # SA1
START = cfg["train_start"]  # 2020-01-01
END = cfg["test_end"]  # 2024-06-30

print(f"Region: {REGION}")
print(f"Window: {START} to {END}")

In [ ]:
prices = load_prices(REGION, START, END, cache=cfg["nemosis_cache"])
demand = load_demand(REGION, START, END, cache=cfg["nemosis_cache"])

---
## 3. Inspect the raw data

Before doing anything with this data, look at it.

In [ ]:
prices.head(12)

In [ ]:
print(f"Shape: {prices.shape}")
print(f"Index: {prices.index.dtype}, freq={prices.index.inferred_freq}")
print(f"Columns: {list(prices.columns)}")
print(f"\nFirst timestamp: {prices.index[0]}")
print(f"Last timestamp:  {prices.index[-1]}")
print(f"Monotonic: {prices.index.is_monotonic_increasing}")
print(f"Unique:    {prices.index.is_unique}")

In [ ]:
prices["price"].describe()

In [ ]:
demand.head(12)

In [ ]:
demand["demand"].describe()

Note the price range: the minimum is deeply negative (generators *paying*
to stay online) and the maximum is extreme (the NEM market price cap is
\$17,500/MWh as of 2024). The mean is modest — most of the time prices
are between \$0 and \$150. The action is in the tails.

---
## 4. Visualise SA1 prices

### 4a. Full history — why you need a symlog axis

Try plotting on a linear axis first. The spikes compress everything
else into a flat line.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Linear scale — spikes dominate
ax1.plot(prices.index, prices["price"], linewidth=0.2, alpha=0.6)
ax1.set_ylabel("Price ($/MWh)")
ax1.set_title(f"{REGION} dispatch price — linear scale")

# Symmetric log scale — structure visible
ax2.plot(prices.index, prices["price"], linewidth=0.2, alpha=0.6)
ax2.set_yscale("symlog", linthresh=100)
ax2.set_ylabel("Price ($/MWh, symlog)")
ax2.set_title(f"{REGION} dispatch price — symlog scale (linthresh=100)")
ax2.axhline(0, color="gray", linewidth=0.5, linestyle="--")

fig.tight_layout()
save_fig(fig, "01_price_history_linear_vs_symlog")
plt.show()

The symlog scale is linear around zero (within `linthresh`) and
logarithmic outside it. Try changing `linthresh` — at 10 you see more
of the low-price structure; at 1000 it looks nearly linear again.

### 4b. Demand

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(demand.index, demand["demand"], linewidth=0.2, alpha=0.6, color="C1")
ax.set_ylabel("Demand (MW)")
ax.set_title(f"{REGION} regional demand")
save_fig(fig, "01_demand_history")
plt.show()

### 4c. One-week zoom

Zoom to a single week to see the **daily cycle**: low overnight, a
morning ramp, a midday dip (rooftop solar pushing net demand down),
and an evening peak when solar drops off and demand stays high.

In [ ]:
week_start, week_end = "2024-01-15", "2024-01-22"
week = prices.join(demand).loc[week_start:week_end]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(week.index, week["price"], "k-", linewidth=0.8)
ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax1.set_ylabel("Price ($/MWh)")
ax1.set_title(f"{REGION} — {week_start} to {week_end}")

ax2.fill_between(week.index, week["demand"], alpha=0.3, color="C1")
ax2.plot(week.index, week["demand"], color="C1", linewidth=0.8)
ax2.set_ylabel("Demand (MW)")

fig.tight_layout()
save_fig(fig, "01_one_week_zoom")
plt.show()

### 4d. Price distribution

A histogram of raw prices is dominated by the bulk near \$50–\$100.
Clip to a sensible range and use log-count to see the tail.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Clipped linear histogram
clipped = prices["price"].clip(-100, 500)
ax1.hist(clipped, bins=200, edgecolor="none", alpha=0.7)
ax1.set_xlabel("Price ($/MWh, clipped to [-100, 500])")
ax1.set_ylabel("Count")
ax1.set_title("Price distribution (clipped)")
ax1.axvline(0, color="red", linewidth=0.8, linestyle="--", label="$0")
ax1.legend()

# Full range with log y-axis
ax2.hist(prices["price"], bins=500, edgecolor="none", alpha=0.7)
ax2.set_yscale("log")
ax2.set_xlabel("Price ($/MWh)")
ax2.set_ylabel("Count (log)")
ax2.set_title("Price distribution (full range, log count)")

fig.tight_layout()
save_fig(fig, "01_price_distribution")
plt.show()

In [ ]:
# Key quantiles
spike_threshold = cfg["spike_threshold_aud"]
n = len(prices)
print(f"Total intervals: {n:,}")
print(f"Negative prices: {(prices['price'] < 0).sum():,} ({100*(prices['price'] < 0).mean():.1f}%)")
print(f"Spikes > ${spike_threshold}: {(prices['price'] > spike_threshold).sum():,} ({100*(prices['price'] > spike_threshold).mean():.2f}%)")
print(f"Missing: {prices['price'].isna().sum():,}")
print("\nPercentiles:")
for q in [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]:
    print(f"  {q:6.0%}: ${prices['price'].quantile(q):>10,.2f}")

---
## 5. Compare all five regions

SA1 is our focus region, but understanding how it compares to the rest
of the NEM is essential context. Let's load all five and compare.

In [ ]:
ALL_REGIONS = ["NSW1", "QLD1", "VIC1", "SA1", "TAS1"]

all_prices = {}
for r in ALL_REGIONS:
    all_prices[r] = load_prices(r, START, END, cache=cfg["nemosis_cache"])
    print(f"{r}: {len(all_prices[r]):,} rows loaded")

### 5a. Map + summary statistics

Side by side: the map shows *where* each region is, the table shows
*what* its prices look like.

In [ ]:
# Summary table for all regions
summary_rows = []
for r in ALL_REGIONS:
    p = all_prices[r]["price"]
    summary_rows.append({
        "Region": r,
        "Mean ($/MWh)": p.mean(),
        "Median": p.median(),
        "Std": p.std(),
        "Min": p.min(),
        "Max": p.max(),
        "% Negative": 100 * (p < 0).mean(),
        f"% > ${spike_threshold}": 100 * (p > spike_threshold).mean(),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Region")
summary_df.round(2)

In [ ]:
# Map coloured by mean price, with key stats annotated
regions_plot = regions_gdf.merge(summary_df, left_on="nem_region", right_index=True)

fig, ax = plt.subplots(figsize=(9, 11))
regions_plot.plot(
    ax=ax, column="Mean ($/MWh)", cmap="YlOrRd",
    edgecolor="white", linewidth=1.5, legend=True,
    legend_kwds={"label": "Mean price ($/MWh)", "shrink": 0.5},
)
for _, row in regions_plot.iterrows():
    ax.annotate(
        f"{row['nem_region']}\n"
        f"${row['Mean ($/MWh)']:.0f} mean\n"
        f"{row['% Negative']:.1f}% neg",
        xy=(row["label_lon"], row["label_lat"]),
        ha="center", fontsize=8, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8),
    )
ax.set_xlim(112, 155)
ax.set_ylim(-45, -10)
ax.set_title("NEM regions — mean dispatch price", fontsize=14)
save_fig(fig, "01_regions_mean_price_map")
plt.show()

### 5b. Price distributions by region

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)

for ax, r in zip(axes, ALL_REGIONS):
    p = all_prices[r]["price"].clip(-100, 500)
    ax.hist(p, bins=150, edgecolor="none", alpha=0.7, color=REGION_COLORS[r])
    ax.set_title(r, fontweight="bold")
    ax.set_xlabel("$/MWh")
    ax.axvline(0, color="red", linewidth=0.5, linestyle="--")

axes[0].set_ylabel("Count")
fig.suptitle("Price distributions by region (clipped to [-100, 500])", fontsize=13)
fig.tight_layout()
save_fig(fig, "01_price_distributions_by_region")
plt.show()

### 5c. Overlaid time series — one week across all regions

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for r in ALL_REGIONS:
    week_r = all_prices[r].loc[week_start:week_end]
    ax.plot(week_r.index, week_r["price"], linewidth=0.8,
            color=REGION_COLORS[r], label=r, alpha=0.8)

ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.set_ylabel("Price ($/MWh)")
ax.set_title(f"All regions — {week_start} to {week_end}")
ax.legend(loc="upper right")
save_fig(fig, "01_all_regions_one_week")
plt.show()

When regions are interconnected and unconstrained, prices converge.
When an interconnector hits its limit, prices separate — that's
**congestion**. Watch for periods where SA1 diverges sharply from VIC1.

### 5d. Negative price frequency by region

In [ ]:
# Monthly negative-price share for each region
fig, ax = plt.subplots(figsize=(14, 5))

for r in ALL_REGIONS:
    monthly_neg = (all_prices[r]["price"] < 0).resample("ME").mean() * 100
    ax.plot(monthly_neg.index, monthly_neg, linewidth=1.2,
            color=REGION_COLORS[r], label=r)

ax.set_ylabel("% of intervals with negative price")
ax.set_title("Monthly negative-price share by region")
ax.legend()
save_fig(fig, "01_negative_price_by_region")
plt.show()

SA1 and QLD1 typically lead on negative prices — both have high
renewable penetration. The seasonal pattern tracks solar output: more
negatives in the sunnier months.

---
## 6. Build and cache the dataset

`build_dataset` joins price and demand, reindexes onto a gapless
5-minute grid (filling any AEMO gaps with NaN), builds a 30-minute
version from the trading price table, and writes both to parquet.

In [ ]:
df_5min, df_30min = build_dataset(
    REGION, START, END,
    cache=cfg["nemosis_cache"],
    out_dir=cfg["paths"]["processed"],
)

print(f"5-min:  {df_5min.shape}, columns: {list(df_5min.columns)}")
print(f"30-min: {df_30min.shape}, columns: {list(df_30min.columns)}")

In [ ]:
# Verify: monotonic, unique, gapless
assert df_5min.index.is_monotonic_increasing, "Not sorted!"
assert df_5min.index.is_unique, "Duplicates!"

expected_5min = pd.date_range(df_5min.index[0], df_5min.index[-1], freq="5min")
assert len(df_5min) == len(expected_5min), (
    f"Gaps! Expected {len(expected_5min):,}, got {len(df_5min):,}"
)
print(f"Gapless 5-min grid: {len(df_5min):,} rows")
print(f"Missing price values: {df_5min['price'].isna().sum():,}")
print(f"Missing demand values: {df_5min['demand'].isna().sum():,}")

### Missingness heatmap

The grid is gapless (every 5-min slot exists), but some slots may have
NaN if AEMO didn't publish data for that interval. A heatmap by date
and time-of-day reveals any patterns.

In [ ]:
missing = df_5min["price"].isna()
if missing.any():
    miss_df = pd.DataFrame({
        "date": missing.index.date,
        "time": missing.index.time,
        "missing": missing.values.astype(int),
    })
    pivot = miss_df.pivot_table(
        index="date", columns="time", values="missing", aggfunc="max"
    )
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(pivot.values, aspect="auto", cmap="Reds", interpolation="nearest")
    ax.set_title(f"{REGION} — missing data (red = NaN)")
    ax.set_ylabel("Date")
    ax.set_xlabel("Time of day")
    n_cols = pivot.shape[1]
    tick_every = max(1, n_cols // 12)
    ax.set_xticks(range(0, n_cols, tick_every))
    ax.set_xticklabels(
        [str(pivot.columns[i]) for i in range(0, n_cols, tick_every)],
        rotation=45,
    )
    save_fig(fig, "01_missingness_heatmap")
    plt.show()
else:
    print("No missing values — the dataset is fully gapless.")

---
## 7. Data dictionary

| Column | Resolution | Unit | AEMO source | Transformation |
|---|---|---|---|---|
| `price` | 5 min | $/MWh | `DISPATCHPRICE.RRP` | Shifted to interval-start |
| `price` | 30 min | $/MWh | `TRADINGPRICE.RRP` | Shifted to interval-start |
| `demand` | 5 min | MW | `DISPATCHREGIONSUM.TOTALDEMAND` | Shifted to interval-start |

**Index:** `timestamp` — the *start* of each interval (not AEMO's raw interval-end).

**Files:**
- `data/processed/SA1_5min.parquet` — 5-minute prices + demand
- `data/processed/SA1_30min.parquet` — 30-minute trading prices + demand

---
## Exercises

### Exercise 1: Trading price vs dispatch price

The 30-minute **trading price** is supposed to be the time-weighted
average of the six 5-minute **dispatch prices** within each trading
interval.

Verify this using the data you have. Where does it hold, and where does
it break? What causes the discrepancies?

<details><summary>Hint 1</summary>

You already have `prices` (5-min dispatch) and `df_30min` (30-min
trading). To average dispatch prices into 30-min buckets, use
`.resample('30min').mean()` — pandas aligns by the index, which is
already interval-start.

</details>

<details><summary>Hint 2</summary>

Compute the difference between the resampled dispatch mean and the
trading price. Most will be near zero. For the outliers, check whether
they coincide with intervention periods — `INTERVENTION=1` rows are
filtered differently in dispatch vs trading tables.

</details>

<details><summary>Hint 3</summary>

Plot a histogram of the differences. Then scatter-plot the absolute
difference against the trading price level — are the discrepancies
concentrated at high prices, low prices, or random?

</details>

<details><summary>Solution</summary>

```python

```

The discrepancies are concentrated at extreme prices and during
intervention periods. The trading price uses a different averaging
method during interventions — it reflects the intervention dispatch
outcome, while our dispatch prices are filtered to `INTERVENTION=0`
(the counterfactual). This is a deliberate choice: for forecasting we
want the market-clearing price, not the administered outcome.

</details>

In [ ]:
# Your analysis here
# Resample 5-min dispatch prices to 30-min means
dispatch_30 = prices["price"].resample("30min").mean()

# Align with trading prices
compare = pd.DataFrame({
    "trading": df_30min["price"],
    "dispatch_mean": dispatch_30,
}).dropna()
compare["diff"] = compare["trading"] - compare["dispatch_mean"]

print(f"Mean difference: ${compare['diff'].mean():.4f}")
print(f"Median difference: ${compare['diff'].median():.4f}")
print(f"Max absolute difference: ${compare['diff'].abs().max():.2f}")
print(f"Intervals with |diff| > $1: {(compare['diff'].abs() > 1).sum():,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(compare["diff"].clip(-50, 50), bins=200, edgecolor="none", alpha=0.7)
ax1.set_xlabel("Trading − Dispatch mean ($/MWh)")
ax1.set_ylabel("Count")
ax1.set_title("Distribution of discrepancies")
ax1.axvline(0, color="red", linewidth=0.8, linestyle="--")

ax2.scatter(compare["trading"], compare["diff"].abs(),
            s=1, alpha=0.3)
ax2.set_xlabel("Trading price ($/MWh)")
ax2.set_ylabel("|Difference| ($/MWh)")
ax2.set_title("Discrepancy vs price level")
ax2.set_yscale("symlog", linthresh=1)

fig.tight_layout()
plt.show()



### Exercise 2: The weight of the tail

What fraction of SA1's total price *variance* is contributed by
intervals where the price exceeds \$300/MWh? These intervals are rare
(check the percentage above), but they may dominate the error of a
model that minimises mean squared error.

What does this imply about the choice of loss function for a price
forecasting model?

<details><summary>Hint 1</summary>

Variance decomposes additively. You can compute the variance
contribution of a subset by looking at the sum of squared deviations
from the overall mean: `((p[mask] - p.mean())**2).sum()` divided by
the total `((p - p.mean())**2).sum()`.

</details>

<details><summary>Hint 2</summary>

Think about what this means for MSE-based models. If 0.5% of
intervals contribute 80% of the variance, an MSE-trained model will
spend most of its capacity on those spikes. Is that what you want, or
would a different target transform (like `arcsinh`) be more useful?

</details>

<details><summary>Hint 3</summary>

Compare the raw price histogram with `np.arcsinh(price)`. How does
the transform redistribute the tail? If variance contribution from
spikes drops from ~80% to ~20%, the model can attend to the rest of
the distribution.

</details>

<details><summary>Solution</summary>

```python
p = prices["price"].dropna()
mean_p = p.mean()
threshold = 300

# Variance decomposition
total_ss = ((p - mean_p) ** 2).sum()
spike_mask = p.abs() > threshold
spike_ss = ((p[spike_mask] - mean_p) ** 2).sum()

print(f"Spike intervals (|price| > ${threshold}): "
      f"{spike_mask.sum():,} ({spike_mask.mean():.2%})")
print(f"Variance contribution: {spike_ss / total_ss:.1%}")

# Same analysis on arcsinh-transformed prices
t = np.arcsinh(p)
mean_t = t.mean()
total_ss_t = ((t - mean_t) ** 2).sum()
spike_ss_t = ((t[spike_mask] - mean_t) ** 2).sum()

print(f"\narcsinh transform:")
print(f"Spike variance contribution: {spike_ss_t / total_ss_t:.1%}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(p.clip(-200, 500), bins=300, edgecolor="none", alpha=0.7)
ax1.axvline(threshold, color="red", linestyle="--", label=f"${threshold}")
ax1.set_xlabel("Price ($/MWh)")
ax1.set_title(f"Raw — spikes = {spike_ss / total_ss:.0%} of variance")
ax1.legend()

ax2.hist(t, bins=300, edgecolor="none", alpha=0.7, color="C1")
ax2.axvline(np.arcsinh(threshold), color="red", linestyle="--",
            label=f"arcsinh({threshold})")
ax2.set_xlabel("arcsinh(price)")
ax2.set_title(f"Transformed — spikes = {spike_ss_t / total_ss_t:.0%} of variance")
ax2.legend()

fig.tight_layout()
plt.show()
```

A tiny fraction of intervals (typically <1%) contributes the
majority of price variance. An MSE-trained model would devote most
capacity to these spikes at the expense of the other 99% of
intervals. The `arcsinh` transform compresses the tails — variance
contribution from spikes drops dramatically, letting the model learn
the base-load pattern. This is why we use arcsinh as our target
transform throughout the curriculum.

</details>

In [ ]:
import numpy as np

# Your analysis here
p = prices["price"].dropna()
mean_p = p.mean()
threshold = 300

# Variance decomposition
total_ss = ((p - mean_p) ** 2).sum()
spike_mask = p.abs() > threshold
spike_ss = ((p[spike_mask] - mean_p) ** 2).sum()

print(f"Spike intervals (|price| > ${threshold}): "
      f"{spike_mask.sum():,} ({spike_mask.mean():.2%})")
print(f"Variance contribution: {spike_ss / total_ss:.1%}")

# Same analysis on arcsinh-transformed prices
t = np.arcsinh(p)
mean_t = t.mean()
total_ss_t = ((t - mean_t) ** 2).sum()
spike_ss_t = ((t[spike_mask] - mean_t) ** 2).sum()

print("\narcsinh transform:")
print(f"Spike variance contribution: {spike_ss_t / total_ss_t:.1%}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(p.clip(-200, 500), bins=300, edgecolor="none", alpha=0.7)
ax1.axvline(threshold, color="red", linestyle="--", label=f"${threshold}")
ax1.set_xlabel("Price ($/MWh)")
ax1.set_title(f"Raw — spikes = {spike_ss / total_ss:.0%} of variance")
ax1.legend()

ax2.hist(t, bins=300, edgecolor="none", alpha=0.7, color="C1")
ax2.axvline(np.arcsinh(threshold), color="red", linestyle="--",
            label=f"arcsinh({threshold})")
ax2.set_xlabel("arcsinh(price)")
ax2.set_title(f"Transformed — spikes = {spike_ss_t / total_ss_t:.0%} of variance")
ax2.legend()

fig.tight_layout()
plt.show()


### Exercise 3: Interconnector congestion

When do SA1 and VIC1 prices diverge? These regions are linked by a
single interconnector (Heywood). When it's unconstrained, the prices
should be similar. When it's congested, they can be very different.

Compute the absolute price difference between SA1 and VIC1 at each
5-minute interval. Plot its distribution and its time series. At what
time of day is congestion most common? In which direction (SA1 higher
or VIC1 higher)?

<details><summary>Hint 1</summary>

Both DataFrames share the same `DatetimeIndex`, so you can align them
with `pd.DataFrame({'SA1': all_prices['SA1']['price'], 'VIC1':
all_prices['VIC1']['price']})`. The result has NaN only where one
region is missing — `.dropna()` handles that.

</details>

<details><summary>Hint 2</summary>

To find the hour-of-day pattern, add the hour as a column:
`df['hour'] = df.index.hour`, then `.groupby('hour')` and compute
the mean absolute difference per hour.

</details>

<details><summary>Hint 3</summary>

For direction: plot `SA1 - VIC1` (not the absolute value). Positive
means SA1 is more expensive. Is this more common in the evening peak
(solar has dropped, SA1 is import-constrained) or midday (SA1 has
excess solar and is export-constrained)?

</details>

<details><summary>Solution</summary>

```python
spread = pd.DataFrame({
    "SA1": all_prices["SA1"]["price"],
    "VIC1": all_prices["VIC1"]["price"],
}).dropna()
spread["diff"] = spread["SA1"] - spread["VIC1"]
spread["abs_diff"] = spread["diff"].abs()
spread["hour"] = spread.index.hour

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of signed spread
axes[0, 0].hist(spread["diff"].clip(-500, 500), bins=300,
                edgecolor="none", alpha=0.7)
axes[0, 0].axvline(0, color="red", linewidth=0.8, linestyle="--")
axes[0, 0].set_xlabel("SA1 − VIC1 ($/MWh)")
axes[0, 0].set_title("Price spread distribution")

# Mean absolute spread by hour of day
hourly = spread.groupby("hour")["abs_diff"].mean()
axes[0, 1].bar(hourly.index, hourly.values, color="C2", alpha=0.7)
axes[0, 1].set_xlabel("Hour of day")
axes[0, 1].set_ylabel("Mean |SA1 − VIC1| ($/MWh)")
axes[0, 1].set_title("Congestion by time of day")

# Mean signed spread by hour — direction matters
hourly_signed = spread.groupby("hour")["diff"].mean()
colors = ["C3" if v > 0 else "C0" for v in hourly_signed.values]
axes[1, 0].bar(hourly_signed.index, hourly_signed.values,
               color=colors, alpha=0.7)
axes[1, 0].axhline(0, color="gray", linewidth=0.5)
axes[1, 0].set_xlabel("Hour of day")
axes[1, 0].set_ylabel("Mean SA1 − VIC1 ($/MWh)")
axes[1, 0].set_title("Direction: red = SA1 higher, blue = VIC1 higher")

# Monthly evolution of mean absolute spread
monthly = spread.resample("ME")["abs_diff"].mean()
axes[1, 1].plot(monthly.index, monthly.values, linewidth=1.2)
axes[1, 1].set_ylabel("Mean |SA1 − VIC1| ($/MWh)")
axes[1, 1].set_title("Monthly congestion trend")

fig.tight_layout()
plt.show()

print(f"Mean absolute spread: ${spread['abs_diff'].mean():.2f}/MWh")
print(f"SA1 higher than VIC1: {(spread['diff'] > 0).mean():.1%} of intervals")
print(f"Spread > $100: {(spread['abs_diff'] > 100).mean():.2%} of intervals")
```

Congestion peaks in the **evening** (hours 17–21) when solar output
drops and SA1 must import from VIC1 through the constrained Heywood
interconnector. During midday, SA1 sometimes has excess solar and the
spread goes negative (SA1 cheaper) — but the magnitude is smaller
because the interconnector is less often constrained in that direction.
The seasonal pattern shows more congestion in summer (high demand from
air conditioning) and during transition months when renewable output
is variable.

</details>

In [ ]:
# Your analysis here
spread = pd.DataFrame({
    "SA1": all_prices["SA1"]["price"],
    "VIC1": all_prices["VIC1"]["price"],
}).dropna()
spread["diff"] = spread["SA1"] - spread["VIC1"]
spread["abs_diff"] = spread["diff"].abs()
spread["hour"] = spread.index.hour

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of signed spread
axes[0, 0].hist(spread["diff"].clip(-500, 500), bins=300,
                edgecolor="none", alpha=0.7)
axes[0, 0].axvline(0, color="red", linewidth=0.8, linestyle="--")
axes[0, 0].set_xlabel("SA1 − VIC1 ($/MWh)")
axes[0, 0].set_title("Price spread distribution")

# Mean absolute spread by hour of day
hourly = spread.groupby("hour")["abs_diff"].mean()
axes[0, 1].bar(hourly.index, hourly.values, color="C2", alpha=0.7)
axes[0, 1].set_xlabel("Hour of day")
axes[0, 1].set_ylabel("Mean |SA1 − VIC1| ($/MWh)")
axes[0, 1].set_title("Congestion by time of day")

# Mean signed spread by hour — direction matters
hourly_signed = spread.groupby("hour")["diff"].mean()
colors = ["C3" if v > 0 else "C0" for v in hourly_signed.values]
axes[1, 0].bar(hourly_signed.index, hourly_signed.values,
               color=colors, alpha=0.7)
axes[1, 0].axhline(0, color="gray", linewidth=0.5)
axes[1, 0].set_xlabel("Hour of day")
axes[1, 0].set_ylabel("Mean SA1 − VIC1 ($/MWh)")
axes[1, 0].set_title("Direction: red = SA1 higher, blue = VIC1 higher")

# Monthly evolution of mean absolute spread
monthly = spread.resample("ME")["abs_diff"].mean()
axes[1, 1].plot(monthly.index, monthly.values, linewidth=1.2)
axes[1, 1].set_ylabel("Mean |SA1 − VIC1| ($/MWh)")
axes[1, 1].set_title("Monthly congestion trend")

fig.tight_layout()
plt.show()

print(f"Mean absolute spread: ${spread['abs_diff'].mean():.2f}/MWh")
print(f"SA1 higher than VIC1: {(spread['diff'] > 0).mean():.1%} of intervals")
print(f"Spread > $100: {(spread['abs_diff'] > 100).mean():.2%} of intervals")


---
## What we learned

1. The NEM is five interconnected regions with 5-minute dispatch prices.
   SA1 is the most volatile — high renewable penetration creates both
   negative prices and extreme spikes.
2. AEMO timestamps are interval-ending; we shift to interval-start once
   in `grian.data` so weather joins are correct.
3. The price distribution has extremely heavy tails: a tiny fraction of
   intervals contribute most of the variance. This will drive our choice
   of target transform (asinh) and loss function in later notebooks.
4. Region comparison reveals interconnector congestion — when regions
   decouple, prices diverge. This is a key signal for forecasting.
5. The dataset is cached as parquet for instant loading going forward.

**Next:** Notebook 02 digs into the statistical structure of this price
series — seasonality, autocorrelation, volatility clustering, and the
stylised facts that any model must respect.

In [ ]:
# Write report
report_dir = Path(cfg["paths"]["reports"])
report_dir.mkdir(parents=True, exist_ok=True)

report = f"""# Notebook 01 — Data Report\n
Region: {REGION} | Window: {START} to {END}\n
## 5-minute dataset\n
- Rows: {len(df_5min):,}\n
- Price range: ${df_5min['price'].min():,.2f} to ${df_5min['price'].max():,.2f}/MWh\n
- Mean price: ${df_5min['price'].mean():,.2f}/MWh\n
- Negative intervals: {(df_5min['price'] < 0).sum():,} ({100*(df_5min['price'] < 0).mean():.1f}%)\n
- Spike intervals (>${spike_threshold}): {(df_5min['price'] > spike_threshold).sum():,}\n
- Missing: {df_5min['price'].isna().sum():,}\n
\n## 30-minute dataset\n
- Rows: {len(df_30min):,}\n
\n## Region comparison\n
"""
for r in ALL_REGIONS:
    p = all_prices[r]["price"]
    report += f"- {r}: mean=${p.mean():.2f}, {100*(p<0).mean():.1f}% neg\n"

(report_dir / "01_data.md").write_text(report)
print("Report written to", report_dir / "01_data.md")